# 第 51 章：Capstone 最终课程包索引与 Part VI 收束清单

这个 notebook 用一个极小的 final-package table，对比“只按最终 readiness 排序”的 naive baseline 和“先检查 closure gate、最终 owner、跨包依赖与 package index”的课程包收束 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_final_package_closure_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['final_readiness_score'] = float(row['final_readiness_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} final-package items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Target audience:', ordered_counts(rows, 'target_audience'))
print('Closure gate:', ordered_counts(rows, 'closure_gate_status'))
print('Final owner:', ordered_counts(rows, 'final_owner_status'))
print('Dependency sync:', ordered_counts(rows, 'dependency_sync_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['package_id']
    for row in rows
    if row['reference_route'] == 'package_ready'
)
top4_readiness = sorted(
    rows,
    key=lambda row: row['final_readiness_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'package_ready'
    for row in top4_readiness
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'package_ready'
    for row in top4_readiness
)
baseline_ready_precision = baseline_ready_hits / len(top4_readiness)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_readiness)

print('Reference package-ready items:', reference_ready_ids)
print('Top-4 by final readiness:')
for row in top4_readiness:
    print(
        f"  {row['package_id']}: readiness={row['final_readiness_score']:.2f}, "
        f"route={row['reference_route']}, scope={row['package_scope']}"
    )
print('Baseline package precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_package_item(row):
    if row['closure_gate_status'] in {'open', 'blocked'}:
        return 'close_blocker_before_closure', 'open or blocked closure gate must be resolved first'
    if row['final_owner_status'] != 'assigned':
        return 'assign_final_owner', 'package has no clear final owner yet'
    if row['dependency_sync_status'] != 'synced':
        return 'sync_cross_package_dependency', 'cross-package links, versions, or shared assets are not synced'
    if row['package_index_status'] != 'complete':
        return 'complete_package_index', 'final package index entry, version, or pointer is not complete enough'
    return 'package_ready', 'package has closed gates, a final owner, synced dependencies, and a complete index entry'


routed_rows = []
for row in rows:
    route, reason = route_package_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Final-package workflow routes:')
for row in routed_rows:
    print(
        f"  {row['package_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'package_ready']
index_queue = [row for row in routed_rows if row['route'] == 'complete_package_index']
blocker_queue = [row for row in routed_rows if row['route'] == 'close_blocker_before_closure']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_final_owner']
dependency_queue = [row for row in routed_rows if row['route'] == 'sync_cross_package_dependency']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'package_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Package-ready queue:', [row['package_id'] for row in ready_queue])
print('Complete-index queue:', [row['package_id'] for row in index_queue])
print('Close-blocker queue:', [row['package_id'] for row in blocker_queue])
print('Assign-owner queue:', [row['package_id'] for row in owner_queue])
print('Sync-dependency queue:', [row['package_id'] for row in dependency_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured package precision:', round(ready_precision, 3))


In [ ]:
def closure_steps(row):
    steps = []
    if row['closure_gate_status'] in {'open', 'blocked'}:
        steps.append('先关闭 blocker 或开放 gate，写清当前阻塞点、负责人与解除条件。')
    if row['final_owner_status'] != 'assigned':
        steps.append('指定最终 owner，并记录签字、维护和下一轮更新责任。')
    if row['dependency_sync_status'] != 'synced':
        steps.append('同步跨包依赖：共享 calendar、rubric、setup、archive 和 release note。')
    if row['package_index_status'] != 'complete':
        steps.append('补完整最终课程包索引：入口、版本、目标读者和最后检查日期。')
    return steps or ['可以进入 final course package；保留版本号、owner 和 closure date。']


for row in routed_rows:
    if row['route'] != 'package_ready':
        print(f"\n{row['package_id']} -> {row['route']} ({row['package_scope']})")
        for step in closure_steps(row):
            print(' -', step)


In [ ]:
final_package_outline = [
    'Bundle name: student, TA, instructor, public, and maintenance package names',
    'Audience: who this bundle is for and what entry point they should use first',
    'Location: PDF, notebook, README, data index, and release note entry point',
    'Version and date: current tag, last check date, and closure date',
    'Owner: final reviewer, maintainer, and next-run handoff owner',
    'Dependency map: shared calendar, rubric, setup guide, archive, and public note',
    'Closure state: ready, blocked, needs sync, needs index, or hold',
]

closure_assistant_prompt = '''你是我的 capstone 最终课程包收束助手。

任务：
1. 先阅读 final-package table，不要只按 final readiness 排序；
2. 先检查 closure gate；
3. 再检查 final owner、dependency sync 和 package index；
4. 把每个 bundle route 到 package_ready、complete_package_index、
   close_blocker_before_closure、assign_final_owner 或 sync_cross_package_dependency；
5. 对每个非 ready bundle 输出收束前 checklist。

输出格式：
- Package-ready bundles
- Complete-index bundles
- Close-blocker bundles
- Assign-owner bundles
- Sync-dependency bundles
- Closure checklist by bundle
'''

print('Final package index outline:')
for item in final_package_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(closure_assistant_prompt)


In [ ]:
closure_snapshot = {
    'dataset': DATA_PATH.name,
    'n_package_items': len(rows),
    'baseline_top4_package_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'package_ready': [row['package_id'] for row in ready_queue],
    'complete_package_index': [row['package_id'] for row in index_queue],
    'close_blocker_before_closure': [row['package_id'] for row in blocker_queue],
    'assign_final_owner': [row['package_id'] for row in owner_queue],
    'sync_cross_package_dependency': [row['package_id'] for row in dependency_queue],
    'python_version': platform.python_version(),
}

print('Final package closure snapshot:')
for key, value in closure_snapshot.items():
    print(f'  {key}: {value}')
